In [ ]:
# All imports and settings in one place, so it's easy to see what the
# notebook depends on and to change a path or the seed once.
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklift.metrics import qini_auc_score, qini_curve

# scikit-uplift 0.5.1 calls a scikit-learn helper that newer versions mark as
# deprecated. The warning is harmless, so hide just that one message.
warnings.filterwarnings("ignore", message=".*stable_cumsum.*", category=FutureWarning)

DATA_PATH = "../data/criteo-uplift-v2.1.csv"
OUTPUT_DIR = "../outputs"
SEED = 42  # used for the sample and the train/test split, so results are reproducible
FEATURES = [f"f{i}" for i in range(12)]  # the 12 anonymized feature columns

TREATED_COLOR = "#2a78d6"
CONTROL_COLOR = "#e34948"

In [ ]:
# One row per user, randomly assigned to treatment (shown an ad) or control.
# The four 0/1 columns are read as int8 instead of the default int64: same
# values, but about 400 MB less memory on 14 million rows.
flag_cols = ["treatment", "conversion", "visit", "exposure"]
df = pd.read_csv(DATA_PATH, dtype={col: "int8" for col in flag_cols})
df.shape

In [3]:
df.head()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [4]:
df.columns

Index(['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10',
       'f11', 'treatment', 'conversion', 'visit', 'exposure'],
      dtype='str')

In [ ]:
# Randomization check: treated/control should look similar across features.
df.groupby("treatment")[FEATURES].mean()

In [6]:
df["treatment"].value_counts()

treatment
1    11882655
0     2096937
Name: count, dtype: int64

In [ ]:
# Conversion, visit and exposure rate for each group, computed in one pass
# instead of filtering the 14M-row table again for every number.
# Row 0 = control, row 1 = treated.
group_rates = df.groupby("treatment")[["conversion", "visit", "exposure"]].mean()
group_rates

In [ ]:
treated_rate = group_rates.loc[1, "conversion"]
control_rate = group_rates.loc[0, "conversion"]
print(treated_rate, control_rate)

In [9]:
# "Naive" to distinguish from the per-user model uplift score computed later.
naive_uplift = treated_rate - control_rate
print(naive_uplift)

0.0011518730521316279


In [ ]:
treated_visit_rate = group_rates.loc[1, "visit"]
control_visit_rate = group_rates.loc[0, "visit"]
uplift_visit = treated_visit_rate - control_visit_rate
print(treated_visit_rate, control_visit_rate, uplift_visit)

In [11]:
df.isnull().sum()

f0            0
f1            0
f2            0
f3            0
f4            0
f5            0
f6            0
f7            0
f8            0
f9            0
f10           0
f11           0
treatment     0
conversion    0
visit         0
exposure      0
dtype: int64

In [12]:
df.describe()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
count,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07,1.397959e+07
mean,1.962030e+01,1.006998e+01,8.446582e+00,4.178923e+00,1.033884e+01,4.028513e+00,-4.155356e+00,5.101765e+00,3.933581e+00,1.602764e+01,5.333396e+00,-1.709672e-01,8.500001e-01,2.916680e-03,4.699200e-02,3.063122e-02
std,5.377464e+00,1.047557e-01,2.993161e-01,1.336645e+00,3.433081e-01,4.310974e-01,4.577914e+00,1.205248e+00,5.665958e-02,7.018975e+00,1.682288e-01,2.283277e-02,3.570713e-01,5.392748e-02,2.116217e-01,1.723164e-01
min,1.261636e+01,1.005965e+01,8.214383e+00,-8.398387e+00,1.028053e+01,-9.011892e+00,-3.142978e+01,4.833815e+00,3.635107e+00,1.319006e+01,5.300375e+00,-1.383941e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.261636e+01,1.005965e+01,8.214383e+00,4.679882e+00,1.028053e+01,4.115453e+00,-6.699321e+00,4.833815e+00,3.910792e+00,1.319006e+01,5.300375e+00,-1.686792e-01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.192341e+01,1.005965e+01,8.214383e+00,4.679882e+00,1.028053e+01,4.115453e+00,-2.411115e+00,4.833815e+00,3.971858e+00,1.319006e+01,5.300375e+00,-1.686792e-01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,2.443646e+01,1.005965e+01,8.723335e+00,4.679882e+00,1.028053e+01,4.115453e+00,2.944427e-01,4.833815e+00,3.971858e+00,1.319006e+01,5.300375e+00,-1.686792e-01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,2.674526e+01,1.634419e+01,9.051962e+00,4.679882e+00,2.112351e+01,4.115453e+00,2.944427e-01,1.199840e+01,3.971858e+00,7.529502e+01,6.473917e+00,-1.686792e-01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


In [ ]:
plt.bar(["Control", "Treated"], [control_rate, treated_rate],
        color=[CONTROL_COLOR, TREATED_COLOR])
plt.title("Conversion rate: treated vs. control")
plt.xlabel("Group")
plt.ylabel("Conversion rate")
plt.savefig(f"{OUTPUT_DIR}/conversion_uplift.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.bar(["Conversion", "Visit"], [naive_uplift, uplift_visit])
plt.title("Conversion vs. visit uplift")
plt.xlabel("Outcome")
plt.ylabel("Uplift (treated rate - control rate)")
plt.savefig(f"{OUTPUT_DIR}/conversionVsvisit.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Conversions are rare - plain accuracy would be a misleading metric here.
# Log scale on the y-axis, otherwise the converters bar is too small to see.
conversion_counts = df["conversion"].value_counts()
plt.bar(["Non-converters", "Converters"],
        [conversion_counts[0], conversion_counts[1]])
plt.yscale("log")
plt.title("Class imbalance: conversions are rare")
plt.xlabel("Outcome")
plt.ylabel("Number of users (log scale)")
plt.savefig(f"{OUTPUT_DIR}/class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
group_counts = df["treatment"].value_counts()
plt.pie([group_counts[1], group_counts[0]],
        labels=["Treated", "Control"],
        colors=[TREATED_COLOR, CONTROL_COLOR],
        autopct="%1.1f%%")
plt.title("Treatment vs. control split")
plt.savefig(f"{OUTPUT_DIR}/treatment_split.png", dpi=150, bbox_inches="tight")
plt.show()

In [17]:
# Data quality check: any exact duplicate rows?
df.duplicated().sum()

np.int64(1259545)

In [ ]:
# Do treated and control look alike on individual features too, not just on
# average? Overlaid density histograms for a few representative features.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["f0", "f3", "f6"]):
    ax.hist(df.loc[df["treatment"] == 1, col], bins=40, density=True,
            alpha=0.6, label="Treated", color=TREATED_COLOR)
    ax.hist(df.loc[df["treatment"] == 0, col], bins=40, density=True,
            alpha=0.6, label="Control", color=CONTROL_COLOR)
    ax.set_title(col)
axes[0].legend()
fig.suptitle("Feature distributions: treated vs. control")
plt.savefig(f"{OUTPUT_DIR}/feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# "treatment" is the random assignment, not proof the ad was actually seen.
# "exposure" tells us who was actually shown an ad - check compliance.
exposure_rate_treated = group_rates.loc[1, "exposure"]
exposure_rate_control = group_rates.loc[0, "exposure"]
print(exposure_rate_treated, exposure_rate_control)

plt.bar(["Treated", "Control"], [exposure_rate_treated, exposure_rate_control],
        color=[TREATED_COLOR, CONTROL_COLOR])
plt.ylabel("Exposure rate")
plt.title("Ad exposure rate by group")
plt.savefig(f"{OUTPUT_DIR}/exposure_compliance.png", dpi=150, bbox_inches="tight")
plt.show()

# EDA Summary

Here's what I found when exploring the data, before doing any modeling.

**The dataset:** about 14 million rows, one row per user. There are 12
features (f0 to f11) that are anonymized, so I don't actually know what they
mean, plus a treatment flag (was this user shown an ad or not) and 3 outcome
columns: conversion, visit, and exposure. No missing values anywhere, which
was nice - nothing to clean up there.

I noticed about 1.26 million rows (roughly 9%) are exact duplicates. That
seemed weird at first, but since the features are anonymized/bucketed
numbers, it makes sense that different users can end up with the same values
by chance. I left them in - didn't drop anything.

**Checking the split is actually fair:** since this is supposed to be a
randomized experiment (some users get ads, some don't), I checked that the
treated and control groups look similar on their features before trusting
any comparison between them. The average feature values are almost
identical between the two groups (e.g. f0 is 19.65 for control vs 19.61 for
treated), and when I plotted the actual distributions
(`outputs/feature_distributions.png`) they overlap almost perfectly. So the
split looks properly random, which means any difference in conversion rate
below should actually be caused by the ad, not by one group just being
different kinds of people to begin with.

**The split itself:** 85% of users were treated (shown an ad), 15% were
control (`outputs/treatment_split.png`). That makes sense - holding people
back from ads costs the company potential sales, so they only keep a small
control group as a baseline.

**Does the ad actually work?** Control group conversion rate: 0.194%.
Treated group: 0.309%. So treated users convert about 60% more often. That's
the basic evidence that advertising has a real, positive effect.

**One thing that surprised me:** "treatment" just means a user was *assigned*
to possibly see an ad - it doesn't mean they actually saw one. When I checked
the `exposure` column, only 3.6% of the treated group was actually shown an
ad (and 0% of control, which makes sense since they were never supposed to
see one). So most of the "treated" group never actually saw the ad, meaning
the 60% number above is being watered down by all the people who were
"treated" but not really exposed. The real effect, for people who actually
see the ad, is probably bigger than this.

In [ ]:
# A 7% sample for speed. Sampling within each (treatment, conversion) group
# keeps the 85/15 split and the rare conversion rate intact.
sample_df = df.groupby(["treatment", "conversion"], group_keys=False).sample(
    frac=0.07, random_state=SEED
)

In [21]:
sample_df.shape

(978571, 16)

In [22]:
sample_df["treatment"].value_counts(normalize=True)

treatment
1    0.850001
0    0.149999
Name: proportion, dtype: float64

In [23]:
sample_df["conversion"].value_counts()

conversion
0    975717
1      2854
Name: count, dtype: int64

In [ ]:
# Stratify again so the test set doesn't end up with too few converters in
# one arm, which would make the Qini evaluation noisy.
train_sample_df, test_sample_df = train_test_split(
    sample_df,
    test_size=0.2,
    random_state=SEED,
    stratify=sample_df[["treatment", "conversion"]],
)

In [25]:
print(train_sample_df.shape, test_sample_df.shape)
print(train_sample_df["treatment"].value_counts(normalize=True))
print(test_sample_df["treatment"].value_counts(normalize=True))

(782856, 16) (195715, 16)
treatment
1    0.850001
0    0.149999
Name: proportion, dtype: float64
treatment
1    0.850001
0    0.149999
Name: proportion, dtype: float64


In [ ]:
# T-learner model 1: trained only on treated users.
treated = train_sample_df[train_sample_df["treatment"] == 1]
X_train_treated = treated[FEATURES]
y_train_treated = treated["conversion"]

In [ ]:
# T-learner model 2: trained only on control users.
control = train_sample_df[train_sample_df["treatment"] == 0]
X_train_control = control[FEATURES]
y_train_control = control["conversion"]

In [29]:
print(X_train_treated.shape, X_train_control.shape)

(665428, 12) (117428, 12)


In [ ]:
# Predicts P(conversion) for a user IF they are treated.
model_treated = LogisticRegression(max_iter=1000)
model_treated.fit(X_train_treated, y_train_treated)

In [31]:
# Predicts P(conversion) for a user IF they are NOT treated.
model_control = LogisticRegression(max_iter=1000)
model_control.fit(X_train_control, y_train_control)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [ ]:
X_test = test_sample_df[FEATURES]

In [33]:
# predict_proba returns [P(0), P(1)]; column 1 is P(conversion=1).
p_treated = model_treated.predict_proba(X_test)[:, 1]
p_control = model_control.predict_proba(X_test)[:, 1]

In [34]:
# T-learner uplift score per user: predicted treated-prob minus control-prob.
uplift = p_treated - p_control

In [35]:
print(uplift[:10])                 # first 10 users' individual uplift
print(uplift.mean())               # average across the test set
print(uplift.min(), uplift.max())  # the full range

[0.00043234 0.00011109 0.0009162  0.00017933 0.00011434 0.00010807
 0.00019423 0.00014746 0.00108834 0.00072385]
0.0009686512818683199
-0.001906923639165307 0.2709246436387698


In [36]:
y_test = test_sample_df["conversion"]
treatment_test = test_sample_df["treatment"]

In [37]:
print(len(y_test), len(treatment_test), len(uplift))

195715 195715 195715


In [39]:
qini = qini_auc_score(y_test, uplift, treatment_test)
print(qini)

0.18660702585380523


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)


In [40]:
x_actual, y_actual = qini_curve(y_test, uplift, treatment_test)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)


In [ ]:
plt.plot(x_actual, y_actual, label="Model")
plt.xlabel("Number of users targeted")
plt.ylabel("Cumulative incremental conversions")
plt.title("Qini curve")
plt.plot([x_actual[0], x_actual[-1]], [y_actual[0], y_actual[-1]],
         linestyle="--", label="Random")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/qini_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Save test-set results for the Streamlit dashboard.
results = test_sample_df[["treatment", "conversion"]].copy()
results["uplift"] = uplift
results.to_csv(f"{OUTPUT_DIR}/uplift_results.csv", index=False)

In [ ]:
plt.hist(results["uplift"], bins=50, color=TREATED_COLOR)
plt.xlabel("Predicted uplift score")
plt.ylabel("Number of users")
plt.title("Distribution of predicted uplift scores (test set)")
plt.savefig(f"{OUTPUT_DIR}/uplift_score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Does a higher predicted uplift decile actually see a bigger realized gap
# between treated and control conversion? This is the key sanity check for
# whether the model's ranking is doing real work.
results["decile"] = pd.qcut(results["uplift"], 10, labels=False, duplicates="drop")

# Conversion rate for every (decile, group) pair, reshaped into a table with
# one row per decile and one column per group (0 = control, 1 = treated).
decile_rates = results.groupby(["decile", "treatment"])["conversion"].mean().unstack()
decile_uplift = decile_rates[1] - decile_rates[0]

# Control converters per decile: when this is small, that decile's bar is noisy.
control_converters = results[results["treatment"] == 0].groupby("decile")["conversion"].sum()
print(pd.DataFrame({"realized_uplift": decile_uplift,
                    "control_converters": control_converters}))

blue_ramp = ["#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6",
             "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
plt.bar(decile_uplift.index.astype(str), decile_uplift.values,
        color=blue_ramp[:len(decile_uplift)])
plt.xlabel("Uplift decile (0 = lowest predicted uplift, highest = best)")
plt.ylabel("Realized uplift (treated - control conversion rate)")
plt.title("Realized uplift by predicted-uplift decile")
plt.savefig(f"{OUTPUT_DIR}/conversion_by_decile.png", dpi=150, bbox_inches="tight")
plt.show()

# How Well Does My Model Work?

Now that I have a model that gives every user an uplift score, I need to
check if that score actually means anything.

**Qini AUC score: 0.187.** This number tells me how much better my model is
at finding the right people to target compared to just picking people at
random. It's clearly above 0, so the model is picking up a real signal - it's
not perfect, but it beats random targeting.

**Qini curve** (`outputs/qini_curve.png`): this plots how many extra
conversions I'd get if I targeted the top X users by my model's score,
compared to targeting X random users instead. My model's line sits above the
random line for most of the chart, which again means it's adding value over
just guessing.

**Conversion rate by decile** (`outputs/conversion_by_decile.png`): I split
users into 10 groups based on their predicted uplift score, from lowest to
highest, and checked the actual gap in conversion rate between treated and
control users in each group. Most of the middle groups (deciles 0-7) are
close to zero and a bit noisy - expected, since conversions are rare and
small groups don't give very stable numbers. But the top group (decile 9,
the users the model is most confident about) has a much bigger uplift than
every other group. So even though the model isn't great at finely ranking
the middle of the pack, it's genuinely good at spotting the small slice of
users who are actually worth targeting - which is really the whole point of
uplift modeling.